<a href="https://colab.research.google.com/github/pavithrahari1395-code/IT-TICKET-ANALYSIS/blob/main/Twitter_A_B_Test_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Twitter A_B testing.csv to Twitter A_B testing.csv


In [ ]:
df = pd.read_csv('Twitter A_B testing.csv')
df['overspend'] = df['campaign_spend'] - df['campaign_budget']
df['overspend_pct'] = (df['overspend'] / df['campaign_budget']) * 100
df['pct_budget_spent'] = (df['campaign_spend'] / df['campaign_budget']) * 100
df['overspend_gt_1pct'] = df['overspend_pct'] > 1

control = df[df['treatment'] == False]
treatment = df[df['treatment'] == True]

In [ ]:
# QUESTION 1: How many campaigns have overspent >1% of their budget?
control_count = control['overspend_gt_1pct'].sum()
treatment_count = treatment['overspend_gt_1pct'].sum()

print(f"Control: {control_count} / {len(control)} = {control_count/len(control)*100:.2f}%")
print(f"Treatment: {treatment_count} / {len(treatment)} = {treatment_count/len(treatment)*100:.2f}%")

# Two-proportion z-test
count = np.array([control_count, treatment_count])
nobs = np.array([len(control), len(treatment)])
z_stat, p_value = proportions_ztest(count, nobs)
print(f"Z-statistic: {z_stat:.4f}, P-value: {p_value:.2e}")


Control: 5716 / 7733 = 73.92%
Treatment: 5180 / 7741 = 66.92%
Z-statistic: 9.5397, P-value: 1.43e-21


In [ ]:
# QUESTION 2: Was treatment effective by company size?
for size in ['small', 'medium', 'large']:
    size_df = df[df['company_size'] == size]
    ctrl = size_df[size_df['treatment'] == False]['overspend_pct']
    trt = size_df[size_df['treatment'] == True]['overspend_pct']

    t_stat, p_val = stats.ttest_ind(ctrl, trt)
    reduction = ctrl.mean() - trt.mean()

    print(f"{size}: Control={ctrl.mean():.2f}%, Treatment={trt.mean():.2f}%, "
          f"Reduction={reduction:.2f}pp, p={p_val:.4f}")

# Two-way ANOVA for interaction
model = ols('overspend_pct ~ C(treatment) * C(company_size)', data=df).fit()
anova_table = anova_lm(model, typ=2)
print(anova_table)


small: Control=37.06%, Treatment=27.10%, Reduction=9.97pp, p=0.0000
medium: Control=4.56%, Treatment=5.14%, Reduction=-0.59pp, p=0.7978
large: Control=12.29%, Treatment=3.26%, Reduction=9.03pp, p=0.0000
                                    sum_sq       df           F         PR(>F)
C(treatment)                  2.911456e+05      1.0   58.073056   2.671706e-14
C(company_size)               2.358415e+06      2.0  235.209439  2.357012e-101
C(treatment):C(company_size)  3.445573e+04      2.0    3.436338   3.220689e-02
Residual                      7.754784e+07  15468.0         NaN            NaN


In [ ]:
# QUESTION 3: Are treatment advertisers entering lower budgets?
control_budget = control['campaign_budget']
treatment_budget = treatment['campaign_budget']

print(f"Control Mean: ${control_budget.mean():,.2f}")
print(f"Treatment Mean: ${treatment_budget.mean():,.2f}")

# T-test
t_stat, p_val = stats.ttest_ind(control_budget, treatment_budget)
print(f"T-test: t={t_stat:.4f}, p={p_val:.4f}")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(control)-1)*control_budget.std()**2 +
                     (len(treatment)-1)*treatment_budget.std()**2) /
                    (len(control)+len(treatment)-2))
cohens_d = (control_budget.mean() - treatment_budget.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.4f}")

# Bootstrap 95% CI
np.random.seed(42)
diffs = [np.random.choice(control_budget.values, len(control_budget), replace=True).mean() -
         np.random.choice(treatment_budget.values, len(treatment_budget), replace=True).mean()
         for _ in range(10000)]
print(f"Bootstrap 95% CI: (${np.percentile(diffs, 2.5):,.2f}, ${np.percentile(diffs, 97.5):,.2f})")

Control Mean: $4,641.83
Treatment Mean: $6,902.23
T-test: t=-1.4197, p=0.1557
Cohen's d: -0.0228
Bootstrap 95% CI: ($-5,775.04, $378.06)


In [ ]:
# QUESTION 4: What is the average overspend percentage?
control_avg = control['overspend_pct'].mean()
treatment_avg = treatment['overspend_pct'].mean()

print(f"Control: {control_avg:.2f}%")
print(f"Treatment: {treatment_avg:.2f}%")
print(f"Reduction: {control_avg - treatment_avg:.2f}pp ({(control_avg - treatment_avg)/control_avg*100:.1f}%)")

# T-test
t_stat, p_val = stats.ttest_ind(control['overspend_pct'], treatment['overspend_pct'])
print(f"T-test: t={t_stat:.4f}, p={p_val:.2e}")

# Mann-Whitney U
u_stat, u_pval = stats.mannwhitneyu(control['overspend_pct'], treatment['overspend_pct'])
print(f"Mann-Whitney U: p={u_pval:.2e}")

Control: 25.32%
Treatment: 17.61%
Reduction: 7.72pp (30.5%)
T-test: t=6.6767, p=2.53e-11
Mann-Whitney U: p=1.01e-23


In [ ]:
# QUESTION 5: How does CTR compare between models?
# DATA LIMITATION: CTR cannot be calculated
print(f"Available columns: {df.columns.tolist()}")
print("CTR requires clicks and impressions data - NOT available in dataset")

Available columns: ['treatment', 'company_size', 'campaign_spend', 'campaign_budget', 'overspend', 'overspend_pct', 'pct_budget_spent', 'overspend_gt_1pct']
CTR requires clicks and impressions data - NOT available in dataset


In [ ]:
# QUESTION 6: What demographic factors correlate with overspending?

# By company size
for size in ['small', 'medium', 'large']:
    size_data = df[df['company_size'] == size]['overspend_pct']
    print(f"{size}: Mean={size_data.mean():.2f}%, Median={size_data.median():.2f}%")

# One-way ANOVA
small = df[df['company_size'] == 'small']['overspend_pct']
medium = df[df['company_size'] == 'medium']['overspend_pct']
large = df[df['company_size'] == 'large']['overspend_pct']
f_stat, p_val = stats.f_oneway(small, medium, large)
print(f"ANOVA: F={f_stat:.2f}, p={p_val:.2e}")

# Budget correlation
corr, p_corr = stats.spearmanr(df['campaign_budget'], df['overspend_pct'])
print(f"Budget-Overspend Correlation: ρ={corr:.4f}, p={p_corr:.2e}")

small: Mean=31.91%, Median=10.00%
medium: Mean=4.84%, Median=2.34%
large: Mean=8.00%, Median=3.32%
ANOVA: F=228.23, p=2.07e-98
Budget-Overspend Correlation: ρ=-0.4830, p=0.00e+00


In [ ]:
# QUESTION 7: How do engagement metrics differ?
# DATA LIMITATION: Engagement metrics not available
print(f"Available columns: {df.columns.tolist()}")
print("Engagement (likes, retweets, replies) - NOT available in dataset")

Available columns: ['treatment', 'company_size', 'campaign_spend', 'campaign_budget', 'overspend', 'overspend_pct', 'pct_budget_spent', 'overspend_gt_1pct']
Engagement (likes, retweets, replies) - NOT available in dataset


In [ ]:
# QUESTION 8: Is % budget spent significantly different?
control_spent = control['pct_budget_spent']
treatment_spent = treatment['pct_budget_spent']

print(f"Control: {control_spent.mean():.2f}%")
print(f"Treatment: {treatment_spent.mean():.2f}%")

# T-test
t_stat, p_val = stats.ttest_ind(control_spent, treatment_spent)
print(f"T-test: t={t_stat:.4f}, p={p_val:.2e}")

# Effect size
pooled_std = np.sqrt(((len(control)-1)*control_spent.std()**2 +
                     (len(treatment)-1)*treatment_spent.std()**2) /
                    (len(control)+len(treatment)-2))
cohens_d = (control_spent.mean() - treatment_spent.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.4f}")



Control: 125.32%
Treatment: 117.61%
T-test: t=6.6767, p=2.53e-11
Cohen's d: 0.1073


In [ ]:
# QUESTION 9: Does company size impact overspend in both groups?
model = ols('overspend_pct ~ C(treatment) + C(company_size) + C(treatment):C(company_size)',
            data=df).fit()
anova_results = anova_lm(model, typ=2)
print(anova_results)


                                    sum_sq       df           F         PR(>F)
C(treatment)                  2.911456e+05      1.0   58.073056   2.671706e-14
C(company_size)               2.358415e+06      2.0  235.209439  2.357012e-101
C(treatment):C(company_size)  3.445573e+04      2.0    3.436338   3.220689e-02
Residual                      7.754784e+07  15468.0         NaN            NaN


In [ ]:
# QUESTION 10: Is variance in campaign spend different between groups?
print(f"Control Variance: {control['campaign_spend'].var():,.2f} (SD: ${control['campaign_spend'].std():,.2f})")
print(f"Treatment Variance: {treatment['campaign_spend'].var():,.2f} (SD: ${treatment['campaign_spend'].std():,.2f})")

# Levene's test (campaign spend)
levene_stat, levene_p = stats.levene(control['campaign_spend'], treatment['campaign_spend'])
print(f"Levene's test (spend): statistic={levene_stat:.4f}, p={levene_p:.4f}")

# Levene's test (overspend %)
lev_os, lev_os_p = stats.levene(control['overspend_pct'], treatment['overspend_pct'])
print(f"Levene's test (overspend %): statistic={lev_os:.4f}, p={lev_os_p:.4f}")

Control Variance: 1,157,680,892.35 (SD: $34,024.71)
Treatment Variance: 7,331,319,045.59 (SD: $85,623.12)
Levene's test (spend): statistic=3.3277, p=0.0681
Levene's test (overspend %): statistic=5.0083, p=0.0252


# Jupyter Notebooks

In this chapter, we'll cover Jupyter Notebooks, including how to write and execute code and how to write text in the **Markdown** format. We'll also discuss what the kernel is, so that you understand generally how Jupyter Notebooks work.

<div class="alert alert-success">
Jupyter notebooks are a way to combine executable code, code outputs, and text into one connected file.
</div>

<div class="alert alert-info">
The official documentation from project Jupyter is available
<a href="https://jupyter-notebook.readthedocs.io/en/stable/" class="alert-link">here</a>
and they also have some example notebooks
<a href="https://github.com/jupyter/notebook/tree/master/docs/source/examples/Notebook" class="alert-link">here</a>
.
</div>

## Menu Options & Shortcuts

To get a quick tour of the Jupyter user-interface, click on the 'Help' menu, then click 'User Interface Tour'.

There are also a large number of useful keyboard shortcuts. Click on the 'Help' menu, and then 'Keyboard Shortcuts' to see a list.

## Cells

<div class="alert alert-success">
    The main organizational structure of the notebook are <b> cells </b>.
</div>

**Cells** are an independent 'unit'. When you click into a cell, you can 'run' it by clicking Shift + Enter, or by pressing the play (Run) button at the top of your notebook.

Cells come in different types for writing different things - mainly, text or code.

### Markdown Cells

Cells, can be plain text. In this book, for chapters where code is included, any place you see text - meaning places were we aren'twriting and executing code  - will be written in **Markdown**.

Markdown is a way to specify all the text formatting you see within the text itself.

#### _Italics_ & __Bold__

For example, *italicized text* can be specified with an \_underscore\_ or \*single asterisks\*. So, using markdown formatting, that text would appear as follows:

For example, italicized text can be specified with an _underscore_ or *single asterisks*.

**Bold text** requires \_\_two underccores\_\_ or \*\*two asterisks\*\* surrounding the text you would like to bold. Using Markdown formatting, that text would look like this:

**Bold text** requires __two underccores__ or **two asterisks** surrounding the text you would like to bold.



#### Headers

There are a number of different size header you can use with Markdown formatting. The number of pound signs preceeding the header specifies the size of the header.

For example, the following:

```
# Headers are specified with a pound sign
## The more pound signs, the smaller the header
#### But it's still larger
than just plain text.
```

would appear as follows using Markdown formatting:


## Headers are specified with a pound sign

### The more pound signs, the smaller the header

#### But it's still larger

than just plain text.

---

Note that to specify text for a header. the pound sign is followed by a space before the text for the header. The largest header -- specified by 1 pound sign -- is referred to as an H1 header. The second largest, H2, and so on and so forth.

#### Lists

Finally, **lists** are also possible with Markdown formatting. Ordered lists are specified with a number, followed by a decimal point, followed by a space and then the text for the list. For example, the following:

```
1. numbered item
2. item 2
3. item 3
```

would appear as follows with Markdown formatting:

1. numbered item
2. item 2
3. item 3

Bulleted or unordered lists are also possible. these are specified by either a dash (`-`) or an asterisk (`*`) instead of the number and decimal point.

For example, either of the following:

```
- item 1
- item 2
- item 3
```

or

```
* item 1
* item 2
* item 3
```

would appear as follows with Markdown formatting:

- item 1
- item 2
- item 3

### Code Cells

Markdown is really helpful when you want to share text within a Jupyter notebook; however, you're here to learn to program. So let's start talking about code! Whenever you're writing code, you'll want to be sure the cell is set to be a code cell. Within a Jupyter notebook, whenever a cell is selected, you can look at the menu across the top. The drop-down menu will specify what type of cell you're working with.

Here, we see an example of what you would see for a Markdown cell.

![Markdown cell specified along toolbar](https://github.com/shanellis/pythonbook/blob/master/content/images/01-intro/markdown_cell.png?raw=1)

To change this cell to a code cell, you would select "Code" from the drop-down menu. You'd then be able to write and execute code from the cell!

![Code selected from drop-down menu](https://github.com/shanellis/pythonbook/blob/master/content/images/01-intro/code_cell.png?raw=1)

The next cell is an example of a code cell. The first line in that cell is a **comment**. Comments are ignored by the computer and are there to help humans reading the code understand what's going on. Comments always start with a pound sign. Note that a pound sign in a Markdown cell indicates something different (a header) than it does in a code cell (a comment).

The second and third line in the example below are lines of Python code! Here we're assigning information to two variables: `a` and `b`. We'll talk all about variable assignment in the next chapter.

In [ ]:
# Cell can also be code.
a = 1
b = 2

#### Running Code

What's important to know now is that Jupyter notebooks don't *just* allow you to write code. You can also **execute the code**. Nothing happens until you execute (or "run") code.

For example, in the first cell above, that code was typed but never executed. We know that it wasn't executed because to the left of the cell we see `In [ ]:`.

![Code cell not yet executed](https://github.com/shanellis/pythonbook/blob/master/content/images/01-intro/not_executed.png?raw=1)

The empty brackets suggested that this code has not yet been executed.

Once executed, a number will show up within those brackets:

![Code cell not yet executed](https://github.com/shanellis/pythonbook/blob/master/content/images/01-intro/executed_code.png?raw=1)

The first code cell you run in a Jupyter notebook will have a `[1]`. The second a `[2]` and so on and so forth.

To execute a cell, you can click on the cell you'd like to run and click "Run" along the toolbar at the top. Or, more likely, you'll want to get in the habit of using hte keyboard shortcut `Shift + Enter` to run your cells. note that `Shift + Enter` will run code cells *and* format Markdown cells.

One important caveat is that if you ever see a `[*]`. to to the left of your code cell, this indicates that the cell is still running. This cann happen if you write code that takes a while to run *or* if you've written code that will run forever (such as an infinite loop).

If this happens and you do *not* want to allow the code to continue to execute (for example, you have an infinite loop and want the code to stop from running, you can click on the square stop icon from the toolbar at the top.

![Stop code cell from executing](https://github.com/shanellis/pythonbook/blob/master/content/images/01-intro/stop.png?raw=1)

Upon executing the code in a cell, often, there will be some sort of output. The code below says subtract 2 (the value stored in `b`) from 1 (the value stored in `a`) and store that in the variable `c`. Then, the final line here says, print that value out to the screen.

In [ ]:
# Cells can also have output, that gets printed out below the cell.
c = a - b
print(c)

-1


What you see above is why notebooks are so great! They allow explanatory text, python code, *and* the outputs from that code to intermingle in a single document!

Before we move on, one more piece of information about Jupyter notebooks in particular. If the last line of code in a code cell is a variable name, the Jupyter notebook, will print the contents of that varaible to your screen, without you having to put `print()` around it. We'll use this a bunch throughout the book, so it's worth noting now, even though we haven't *exactly* covered what variables are...yet!

In [ ]:
# if variable last thing in cell
# output will be variable conents
c

-1

#### Cell Order

Ok, so we've discussed that the numbers in the square brackets to the left of a cell show which cells have been run, and in what order and that an asterisk (`*`) means that the cell is currently running. What we haven't yet discussed is that the order in which the cells are run does ***not*** matter to Python or Jupyter notebooks.

This allows you to flexibly test and develop code. For example, say you've run a few cells of code in order from top to bottom. Then, you realize you wanted a different value in that first code cell. You can go back to that first cell, change it to be the value you want, and then return back to a different cell. Python and Jupyter Notebooks will keep track of whatever was run most recently.

For beginners and individuals less familiar with working in notebooks, this can sometimes take a bit to get used to, but the benefits of this flexibility outweigh the cognitive load it takes to remember that the order isn't set in stone to go from top to bottom.

## Accessing Documentation

When you're new to Python (or any programming language), knowing where to look for more information is critical. While Google and StackOverflow will likely be very helpful, Python and Jupyter Notebooks have built-in ways for you to access documentation that will provide you with helpful information.

<div class="alert alert-success">
Jupyter has useful shortcuts. Add a single '?' after a function or class get a window with the documentation, or a double '??' to pull up the source code.
</div>

In [ ]:
# For example, execute this cell to see the documentation for the 'abs'
abs?

## Autocomplete

<div class="alert alert-success">
Jupyter also has
<a href="https://en.wikipedia.org/wiki/Command-line_completion" class="alert-link">tab complete</a>
capacities, which can autocomplete what you are typing, and/or be used to explore what code is available.  
</div>

In [ ]:
# Move your cursor to the end of the line, press tab, and a drop menu will appear showing all possible completions
ra

In [ ]:
# If there is only one option, tab-complete will auto-complete what you are typing
ran

## Web Browser

<div class="alert alert-success">
Jupyter notebooks display in a web browser. They are not hosted on the web, everything is happening locally.
</div>

If you click on the url in the browser, you will notice it says 'localhost'. This means it is connected to something locally, on your computer.

That local connection is to the 'kernel'.

## Kernels

<div class="alert alert-success">
The 'kernel' is the thing that executes your code. It is what connects the notebook (as you see it) with the part of your computer that runs code.
</div>

Your kernel also stores your **namespace** - all the variables and code that you have declared (executed).

It can be useful to clear and re-launch the kernel. You can do this from the 'kernel' drop down menu, at the top, optionally also clearing all ouputs. Note that this will erase any variables that are stored in memory.

<div class="alert alert-info">
For more useful information, check out Jupyter Notebooks
<a href="https://www.dataquest.io/blog/jupyter-notebook-tips-tricks-shortcuts/" class="alert-link">tips & tricks</a>
, and more information on how
<a href="http://jupyter.readthedocs.io/en/latest/architecture/how_jupyter_ipython_work.html" class="alert-link">notebooks work</a>.
</div>

## Exercises

Q1. **What does three underscores around text accomplish?**  

A) bold  
B) italicize  
C) bold + italicize  
D) plain text    


Q2. **Add a cell to a notebook, change it to be a Markdown cell, and add your name as (1) plain text, (2) italicized text, and (3) bold text.**
  

Q3. **What would happen if I specified a numbered list but put the same number before each list item?**  

A) the list would have the same number before each item  
B) markdown would still format it with sequential numbers  
C) markdown wouldn't know it was a list  
D) normal text with everything on a single line  


Q4. **Write code in a code cell and execute it so that the value returned from the cell is '6'.**


Q5. **If the following were in a code cell, what would be the output of the following?**

```python
a = 1
b = 2
c = 3
print(a + b + c)
```
  
Q6. **If the following were in a code cell, what would be the output of the following?**

```python
a = 1
b = 2
c = 3
a + b + c
```
  
Q7. **If the following were in a code cell, what would be the output of the following?**

```python
a = 1
b = 2
c = 3
```